# P2.1 — Semantic Expert FINAL (Kaggle)

This notebook is preconfigured for the frozen P2.1 final protocol. It uses source-only, label-free reconstruction data and does not claim anomaly-detection performance.

Before running:
1. Attach the Kaggle dataset containing the extracted `phase2-final` and `seqlogad-code-final` folders. Do not enter their paths manually.
2. In **Session options**, enable an NVIDIA GPU and **Internet**. Native-BF16 hardware is preferred; T4 FP16 is conditional on the finite preflight.
3. Add a Kaggle User Secret named `HF_TOKEN` and grant this notebook access. Never paste the token into a cell.
4. Run all cells from the top. Inputs stay read-only under `/kaggle/input`; code, venv, checkpoints and exports are written only under `/kaggle/working`.
5. Change only `FOLD_ID`, `SEED`, or `RESUME_FROM` for another frozen final run.

The notebook verifies the extracted code provenance and data bundle hash, copies code to a writable working directory, and supports both the original final code dataset and the RUN_ID-fixed revision.


In [13]:
# CONFIG — frozen final protocol; change only FOLD_ID, SEED, or RESUME_FROM.
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = str(WORK_ROOT / "seqlogad_outputs")
DATA_ROOT = None       # discovered from the trusted bundle hash
REPO_ROOT = None       # copied from verified Kaggle input to /kaggle/working
FOLD_ID = "FOLD-TARGET-ARCH-BGL"
RUN_MODE = "final"
SEED = 42              # allowed: 42, 3407, 8675309
RESUME_FROM = None     # path to an intact checkpoint, if resuming

TARGET = FOLD_ID.removeprefix("FOLD-TARGET-ARCH-")
RUN_ID = f"P2.1-FINAL-{TARGET}-S{SEED}"
TRAINING_MODE = "QLORA_NF4_DOUBLE_QUANT"
HYPERPARAMETERS = {"seed": SEED, "training_mode": TRAINING_MODE}
EXPECTED_BUNDLE_SHA256 = "3ef2bfe36d2ad68205eea4290a632eb81db0d63acfb76c92f25a3eb96e8205de"
EXPECTED_CODE_PROVENANCE_SHA256S = {
    "0e9adac595fa64acc07be66fadaad010c2af4d57ffcc6baf467bf10ce93f6ff0",  # original final package
    "325299b865d4bfc1c7bf45df3e48544de1194c3452088bcfa09970150a8befef",  # RUN_ID-fixed package
}
assert RUN_MODE == "final"
assert FOLD_ID in {"FOLD-TARGET-ARCH-HDFS", "FOLD-TARGET-ARCH-BGL", "FOLD-TARGET-ARCH-HADOOP"}
assert SEED in {42, 3407, 8675309}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def run_live(args, check=True):
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env) as process:
        try:
            for line in process.stdout:
                token = os.environ.get("HF_TOKEN")
                if token:
                    line = line.replace(token, "[TOKEN_HIDDEN]")
                print(re.sub(r"hf_[A-Za-z0-9]+", "[TOKEN_HIDDEN]", line), end="", flush=True)
            code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
            raise
    if code and check:
        raise RuntimeError(f"Command failed (exit {code}); read the actual error above.")
    return code


## Discover trusted inputs and construct the isolated runtime

Kaggle datasets are already extracted and are read-only. This cell locates the unique matching bundle/code tree, verifies it, copies code into `/kaggle/working`, applies the historical RUN_ID compatibility repair only when needed, and creates the frozen Python 3.12 environment.


In [14]:
assert KAGGLE_INPUT_ROOT.is_dir(), (
    "This notebook must run on Kaggle with the DACNTT dataset attached"
)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. Locate and verify the Phase-2 data
# ============================================================

bundle_candidates = sorted(
    KAGGLE_INPUT_ROOT.rglob("manifests/bundle.json")
)

bundle_matches = [
    path
    for path in bundle_candidates
    if sha256_file(path) == EXPECTED_BUNDLE_SHA256
]

assert len(bundle_matches) == 1, (
    f"Expected exactly one phase2 bundle with SHA-256 "
    f"{EXPECTED_BUNDLE_SHA256}; found {len(bundle_matches)} "
    f"among {[str(path) for path in bundle_candidates]}"
)

DATA_ROOT = str(bundle_matches[0].parent.parent)

# ============================================================
# 2. Locate and verify the extracted code package
# ============================================================

config_candidates = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "configs/models/base-freeze-v1.yaml"
    )
)

verified_code_roots = []

for config_path in config_candidates:
    root = config_path.parents[2]

    provenance_path = root / "CODE_PROVENANCE.json"
    train_path = root / "src/seqlogad/semantic/train.py"

    if not provenance_path.is_file():
        continue

    if not train_path.is_file():
        continue

    provenance_sha256 = sha256_file(provenance_path)

    if provenance_sha256 not in EXPECTED_CODE_PROVENANCE_SHA256S:
        continue

    provenance = json.loads(provenance_path.read_text())
    expected_files = provenance["files"]

    actual_files = {
        path.relative_to(root).as_posix(): sha256_file(path)
        for path in root.rglob("*")
        if path.is_file()
        and path.name != "CODE_PROVENANCE.json"
    }

    if actual_files == expected_files:
        verified_code_roots.append(root)

assert len(verified_code_roots) == 1, (
    f"Expected exactly one verified SeqLogAD code tree; "
    f"found {len(verified_code_roots)}. "
    f"Candidates: {[str(path) for path in config_candidates]}"
)

SOURCE_REPO_ROOT = verified_code_roots[0]

# ============================================================
# 3. Copy read-only Kaggle code into writable working storage
# ============================================================

working_repo = WORK_ROOT / "seqlogad_code"
working_marker = (
    working_repo / ".seqlogad_kaggle_working_copy"
)

if working_repo.exists():
    assert working_marker.is_file(), (
        f"Refusing to replace unrecognized directory: "
        f"{working_repo}"
    )
    shutil.rmtree(working_repo)

shutil.copytree(SOURCE_REPO_ROOT, working_repo)

working_marker.write_text(
    "generated from verified Kaggle input\n"
)

REPO_ROOT = str(working_repo)

BASE_FREEZE_CONFIG = str(
    working_repo / "configs/models/base-freeze-v1.yaml"
)

# ============================================================
# 4. Repair RUN_ID validator in old code package if necessary
# ============================================================

train_py = (
    working_repo
    / "src/seqlogad/semantic/train.py"
)

source = train_py.read_text()

old_guard = """    if not run_id or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_' for c in run_id):
        raise ValueError('unsafe run_id')
"""

fixed_guard = """    import re
    if not isinstance(run_id, str) or re.fullmatch(
            r'[A-Za-z0-9]+(?:[._-][A-Za-z0-9]+)*', run_id) is None:
        raise ValueError('unsafe run_id')
"""

if old_guard in source:
    train_py.write_text(
        source.replace(old_guard, fixed_guard, 1)
    )
    run_id_guard_status = "PATCHED_IN_WORKING_COPY"

elif (
    "RUN_ID_PATTERN = re.compile" in source
    and "validate_run_id(run_id)" in source
):
    run_id_guard_status = "ALREADY_FIXED"

else:
    raise RuntimeError(
        "Unrecognized RUN_ID validator; "
        "refusing to modify code"
    )

compile(
    train_py.read_text(),
    str(train_py),
    "exec",
)

# ============================================================
# 5. Resolve the frozen Python 3.12 runtime
# ============================================================

python312 = (
    sys.executable
    if sys.version_info[:2] == (3, 12)
    else shutil.which("python3.12")
)

if not python312:
    run_live(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "uv",
        ],
        check=True,
    )

    uv = shutil.which("uv")

    assert uv, (
        "uv installation succeeded but executable "
        "was not found on PATH"
    )

    run_live(
        [
            uv,
            "python",
            "install",
            "3.12",
        ],
        check=True,
    )

    python312 = subprocess.check_output(
        [
            uv,
            "python",
            "find",
            "3.12",
        ],
        text=True,
    ).strip()

version_probe = subprocess.run(
    [
        python312,
        "-c",
        "import platform; print(platform.python_version())",
    ],
    text=True,
    capture_output=True,
)

assert version_probe.returncode == 0, (
    version_probe.stderr
)

python_version = version_probe.stdout.strip()

assert python_version.startswith("3.12."), (
    f"Expected Python 3.12, got {python_version}"
)

# ============================================================
# 6. Create the venv and install Kaggle compatibility first
# ============================================================

VENV = WORK_ROOT / "seqlogad-venv"
PYTHON = str(VENV / "bin/python")

run_live(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "virtualenv==20.31.2",
    ],
    check=True,
)

run_live(
    [
        sys.executable,
        "-m",
        "virtualenv",
        "--python",
        python312,
        str(VENV),
    ],
    check=True,
)

# Kaggle sitecustomize imports wrapt.
# This must run immediately after creating the venv.
run_live(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "wrapt>=1.16,<2",
    ],
    check=True,
)

run_live(
    [
        PYTHON,
        "-c",
        (
            "import wrapt; "
            "print('wrapt:', wrapt.__version__); "
            "print('KAGGLE WRAPT READY')"
        ),
    ],
    check=True,
)

run_live(
    [
        PYTHON,
        "-m",
        "pip",
        "--version",
    ],
    check=True,
)

# ============================================================
# 7. Install the frozen CUDA/PyTorch and SeqLogAD runtime
# ============================================================

run_live(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "torch==2.9.1",
        "--index-url",
        "https://download.pytorch.org/whl/cu128",
    ],
    check=True,
)

run_live(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "-e",
        REPO_ROOT,
        "packaging>=24,<27",
    ],
    check=True,
)

run_live(
    [
        PYTHON,
        "-c",
        (
            "from seqlogad.semantic.runtime "
            "import install_missing; "
            f"install_missing({REPO_ROOT!r})"
        ),
    ],
    check=True,
)

# ============================================================
# 8. Resolve and print only non-secret frozen identities
# ============================================================

resolved = subprocess.check_output(
    [
        PYTHON,
        "-c",
        (
            "import json; "
            "from seqlogad.semantic.config "
            "import load_config; "
            f"print(json.dumps("
            f"load_config({REPO_ROOT!r}, "
            f"run_mode='final')[0]))"
        ),
    ]
)

FROZEN = json.loads(resolved)

MODEL_ID = FROZEN["model_id"]
MODEL_REVISION = FROZEN["model_revision"]
TOKENIZER_REVISION = FROZEN[
    "tokenizer_revision"
]

print(
    {
        "DATA_ROOT": DATA_ROOT,
        "SOURCE_REPO_ROOT": str(
            SOURCE_REPO_ROOT
        ),
        "REPO_ROOT": REPO_ROOT,
        "OUTPUT_ROOT": OUTPUT_ROOT,
        "python": python_version,
        "run_id_guard": run_id_guard_status,
        "MODEL_ID": MODEL_ID,
        "MODEL_REVISION": MODEL_REVISION,
        "TOKENIZER_REVISION": (
            TOKENIZER_REVISION
        ),
        "kaggle_wrapt": "READY",
    }
)

created virtual environment CPython3.12.13.final.0-64 in 198ms
  creator CPython3Posix(dest=/kaggle/working/seqlogad-venv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: accelerate==1.12.0, annotated_types==0.8.0, bitsandbytes==0.49.2, cachetools==4.2.1, certifi==2026.7.22, charset_normalizer==3.5.1, drain3==0.9.11, filelock==3.32.3, fsspec==2026.7.0, [TOKEN_HIDDEN]==1.6.0, huggingface_hub==0.36.2, idna==3.19, jinja2==3.1.6, jsonpickle==1.5.1, markupsafe==3.0.3, mpmath==1.3.0, networkx==3.6.1, numpy==2.5.3, nvidia_cublas_cu12==12.8.4.1, nvidia_cuda_cupti_cu12==12.8.90, nvidia_cuda_nvrtc_cu12==12.8.93, nvidia_cuda_runtime_cu12==12.8.90, nvidia_cudnn_cu12==9.10.2.21, nvidia_cufft_cu12==11.3.3.83, nvidia_cufile_cu12==1.13.1.3, nvidia_curand_cu12==10.3.9.90, nvidia_cusolver_cu12==11.7.3.90, nvidia_cusparse_cu12==12.5.8.93, nvidia_cusparselt_cu12==0.7.1, nvidia_nc

## GPU/CUDA/VRAM and frozen runtime check

This check must pass before authentication or model loading. Kaggle may assign different GPU types; FP16 hardware is accepted only if the later forward/backward preflight is finite.


In [15]:
def command(action, *extra):
    return [PYTHON, "-m", "seqlogad.semantic.cli", action, "--repo-root", REPO_ROOT,
            "--data-root", DATA_ROOT, "--output-root", OUTPUT_ROOT, "--fold-id", FOLD_ID,
            "--run-id", RUN_ID, "--run-mode", RUN_MODE, *extra]

run_live(command("environment"), check=True)


{
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "packages": {
    "tqdm": "4.70.1",
    "charset-normalizer": "3.5.1",
    "pydantic": "2.13.5",
    "idna": "3.19",
    "networkx": "3.6.1",
    "nvidia-nvtx-cu12": "12.8.90",
    "nvidia-nvshmem-cu12": "3.3.20",
    "packaging": "26.3",
    "psutil": "7.2.2",
    "nvidia-cusparse-cu12": "12.5.8.93",
    "MarkupSafe": "3.0.3",
    "filelock": "3.32.3",
    "typing_extensions": "4.16.0",
    "pydantic_core": "2.46.5",
    "Jinja2": "3.1.6",
    "nvidia-nccl-cu12": "2.27.5",
    "safetensors": "0.7.0",
    "polars-runtime-32": "1.44.2",
    "transformers": "4.57.6",
    "peft": "0.18.1",
    "nvidia-nvjitlink-cu12": "12.8.93",
    "polars": "1.44.2",
    "nvidia-cublas-cu12": "12.8.4.1",
    "requests": "2.34.2",
    "pip": "25.1.1",
    "cachetools": "4.2.1",
    "huggingface_hub": "0.36.2",
    "torch": "2.9.1+cu128",
    "bitsandbytes": "0.49.2",
    "seqlogad": "0.1.0",
    "nvidia-cusolver-cu12": "11.7

0

## Kaggle Secret authentication and frozen revision verification

Create the `HF_TOKEN` secret in Kaggle and enable notebook access. Its value is never printed or saved.


In [16]:
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        raise RuntimeError(
            "Add a Kaggle User Secret named HF_TOKEN and grant this notebook access"
        ) from None
assert os.environ.get("HF_TOKEN"), "HF_TOKEN Kaggle Secret is empty"
run_live(command("auth"), check=True)


{
  "authentication": "PASS",
  "revision": "d04e592bb4f6aa9cfee91e2e20afa771667e1d4b",
  "metadata": "PASS"
}


0

## Manifest, checksum and leakage validation

The attached data remains under read-only `/kaggle/input`. Validation checks the trusted bundle digest, complete allowlist, hashes, source-only roles, Phase-1 membership and forbidden fields. No anomaly labels are used.


In [17]:
assert Path(DATA_ROOT, "manifests/bundle.json").is_file()
run_live(command("validate", "--expected-bundle-sha256", EXPECTED_BUNDLE_SHA256), check=True)
OVERRIDES = str(WORK_ROOT / "semantic-run-config.json")
Path(OVERRIDES).write_text(json.dumps(HYPERPARAMETERS))
print({
    "DATA_ROOT": DATA_ROOT,
    "OUTPUT_ROOT": OUTPUT_ROOT,
    "FOLD_ID": FOLD_ID,
    "RUN_ID": RUN_ID,
    "hyperparameters": HYPERPARAMETERS,
    "resume": RESUME_FROM,
})


{
  "status": "PASS",
  "fold_id": "FOLD-TARGET-ARCH-BGL",
  "counts": {
    "SOURCE_TRAIN": {
      "ARCH-HDFS": 4746575,
      "ARCH-HADOOP": 237429,
      "ARCH-OPENSTACK": 103898
    },
    "SOURCE_VALIDATION": {
      "ARCH-HDFS": 1025895,
      "ARCH-HADOOP": 110110,
      "ARCH-OPENSTACK": 31135
    }
  },
  "bundle_sha256": "3ef2bfe36d2ad68205eea4290a632eb81db0d63acfb76c92f25a3eb96e8205de",
  "view": {
    "builder_code_sha256": {
      "__init__.py": "0683c49852c76026893ae55a5a908d6765cb36433ed6c03f565a4da6a66a3e6b",
      "checkpoint.py": "0b4847199a7debbc9bab652505848d054b832e61ab34772b2a08c34fc6421773",
      "cli.py": "0b5cb67a6a5db632eea24644df0c9406e2b662c9c0c9aca1125dccd372f74c8e",
      "config.py": "1b3d0972ecc11b4d088b64c89d49a7e9be6257893446d8ee15f04cfec8a9ac93",
      "contracts.py": "47ceff883bb5fcb7f9a2834e9f8b451ee11b1dde634e72e66403e0808ed70211",
      "expert.py": "67b3c57a443efc5270c4eea99277fdf19012728ee0081fc348b9ff5af298e75e",
      "model.py": "d748b0a53c

## Final QLoRA training

The Python modules resolve the frozen final budget: 8192 train and 512 validation records per source, maximum 1536 steps, seed-specific deterministic selection, validation/checkpoint every 100 steps, and early stopping on aggregate validation reconstruction NLL with patience 3 and minimum delta 0.001. No anomaly labels are used.

The first real model step runs a finite forward/backward preflight. Any OOM or non-finite loss aborts rather than silently changing the scientific configuration.


In [ ]:
assert Path(DATA_ROOT, "manifests/bundle.json").is_file(), "Run data validation first"
run_directory = Path(OUTPUT_ROOT) / FOLD_ID / "SEMANTIC_LLAMA" / RUN_ID
assert not run_directory.exists(), f"Run directory already exists: {run_directory}"
args = command("train", "--config-json", OVERRIDES,
               "--expected-bundle-sha256", EXPECTED_BUNDLE_SHA256)
if RESUME_FROM:
    args += ["--resume-from", RESUME_FROM]
run_live(args, check=True)


`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 4 files: 100%|██████████| 4/4 [01:02<00:00, 15.59s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:41<00:00, 10.44s/it]
/kaggle/working/seqlogad-venv/lib/python3.12/site-packages/transformers/models/llama/modeling_llama.py:101: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:304.)
  freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
/kaggle/working/

## Selected checkpoint, metrics and sanity evidence

These are reconstruction and engineering sanity outputs, not anomaly-detection performance.


In [21]:
RUN_DIR = Path(OUTPUT_ROOT) / FOLD_ID / "SEMANTIC_LLAMA" / RUN_ID
for name in ["metrics.json", "coverage.json", "selection.json", "sanity-evidence.json",
             "semantic-probes.json", "manifest.json", "checksums.sha256"]:
    path = RUN_DIR / name
    print(f"\n--- {name} ---")
    if name.endswith(".json"):
        print(json.dumps(json.loads(path.read_text()), indent=2))
    else:
        print(path.read_text())



--- metrics.json ---
{
  "best": {
    "checkpoint": "step-800",
    "loss": 0.02945910496899053
  },
  "early_stopping": {
    "minimum_delta": 0.001,
    "patience": 3,
    "stale_evaluations": 3,
    "triggered": true
  },
  "gradient_norm": 0.015925046056509018,
  "loss_scale": 1.0,
  "optimizer_update_applied": true,
  "peak_vram_allocated": 9482792448,
  "peak_vram_reserved": 11741954048,
  "per_source_validation_reconstruction_nll": {
    "ARCH-HADOOP": 0.056750297123416504,
    "ARCH-HDFS": 0.025252742673174566,
    "ARCH-OPENSTACK": 0.008029084757026794
  },
  "runtime_seconds": 24125.062453788,
  "source_validation_reconstruction_nll": 0.029993288256891813,
  "step": 1100,
  "train_reconstruction_nll": 0.00028728295256996716
}

--- coverage.json ---
{
  "train_mean_coverage": 1.0,
  "train_rejected": [
    {
      "architecture_id": "ARCH-HADOOP",
      "rank": 81193,
      "reason": "empty semantic text: abstain"
    },
    {
      "architecture_id": "ARCH-HADOOP",
      "r

## Export the complete run

The ZIP remains in `/kaggle/working`, appears under notebook Output after **Save Version**, and can also be downloaded from the link displayed below. Preserve the entire run, including all checkpoint siblings and ledgers.


In [22]:
from IPython.display import FileLink, display

archive = shutil.make_archive(str(WORK_ROOT / "seqlogad_outputs"), "zip", OUTPUT_ROOT)
print({"export": archive, "sha256": sha256_file(archive)})
display(FileLink(archive))


KeyboardInterrupt: 